In [1]:
from sage.libs.ecl import ecl_eval
ecl_eval("(ext:set-limit 'ext:heap-size 0)")   # 0 = no ECL-side limit

<ECL: 0>

In [2]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz z')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz,
 z)

In [3]:
filename = "NLO_e0/et_e0_NLO_3PN.txt"

with open(filename, "r") as f:
    content = f.read()

# Extract the part after "dt_du_final ="
import re
match = re.search(r"et_e0_NLO_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    # Convert string back into a Sage expression
    et_e0_NLO_3PN = (sage_eval(expr_str, locals=globals())).subs({SO^2:0},{SO^3:0})

In [4]:
import re
filename = "../x_e_files/x_3PN.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"x_3PN\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    xe_expr = (((sage_eval(expr_str, locals=globals()))*(e_t^(12/19)/e0^(12/19))).expand()).subs({SO^2:0},{SO^3:0})

In [5]:
progress_file = "NNLO_e0/progress.txt"
count = 0

def update_progress(step_name=""):
    global count
    count += 1
    with open(progress_file, "w") as f:
        f.write("# Expansion of count\n")
        f.write("# Plain ASCII output from Sage\n\n")
        f.write(f"count = {count}\n")
        if step_name:
            f.write(f"step = {step_name}\n")

**N**

In [6]:
xe_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_NLO_3PN.coefficient(Xi,0)})

In [7]:
et_e0_N=(((((((xe_N))^(19/12))).taylor(e0,0,5))))*(e0/chi^(19/18))

In [8]:
filename = "NNLO_e0/et_e0_NNLO_N.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_N\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_N =\n")
    f.write(str(et_e0_N.coefficient(Xi,0)))
    f.write("\n\n")

**1 PN**

In [9]:
xe_1PN_expr=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_NLO_3PN.coefficient(Xi,2)*Xi^2})


In [10]:
xe_1PN_1=xe_1PN_expr*Xi^2*x0+xe_N
et_e0_1PN=((((xe_1PN_1)^(19/12))).taylor((Xi,0),(e0,0),7))*(e0/chi^(19/18))


In [11]:
filename = "NNLO_e0/et_e0_NNLO_1PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_1PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_1PN =\n")
    f.write(str(et_e0_1PN.coefficient(Xi,2)*Xi^2))
    f.write("\n\n")

**N and 1PN again, with truncated power series (extra, for comparison)**

The cells above are the original method and are unchanged. The cells below redo N and 1PN
without symbolic `.taylor`:

* Everything lives in a polynomial ring in `e0`, `Xi` whose coefficients are polynomials in
  `c = chi^(1/18)`, `eta`, `x0` (exact rationals). After every multiplication, terms of total
  degree above the target in (`e0`, `Xi`) are dropped (`trunc`), the same cut as
  `.taylor((Xi,0),(e0,0),N)`, so intermediate expressions never grow.
* `e_t^b` (fractional `b`) is written as `e0^b * u^b` with `u = e_t/e0 = c^(-19)*(1+delta)`, and
  `(1+delta)^b`, like the final `(...)^(19/12)`, is a truncated binomial series. In each term of
  `xe_expr` the powers of `e0` and `e_t` add up to a whole number, so only integer powers of `e0` remain.
* The last cells check the results are **exactly** equal to `et_e0_N` and `et_e0_1PN` above.

In [12]:
# ---------- power-series machinery (fast replacement for symbolic .taylor) ----------
import operator
from sage.symbolic.operators import add_vararg, mul_vararg
ADD = (operator.add, add_vararg); MUL = (operator.mul, mul_vararg)

# Coefficient ring: Laurent polynomials in c = chi^(1/18), eta, x0 (exact rationals)
Rc = LaurentPolynomialRing(QQ, 'c, eta_r, x0_r')
c_r, eta_r, x0_r = Rc.gens()
# Series ring: polynomials in e0, Xi over Rc, truncated by hand (see trunc)
P = PolynomialRing(Rc, 'e0_s, Xi_s')
e0_s, Xi_s = P.gens()

SYM2P = {e0: e0_s, Xi: Xi_s, eta: P(eta_r), x0: P(x0_r)}

def keep(m, N):
    """Is monomial e0^m[0] * Xi^m[1] kept at order N?
    N = integer        -> total degree <= N           (like .taylor((Xi,0),(e0,0),N))
    N = (Ne0, NXi)     -> e0 power <= Ne0 AND Xi power <= NXi (separate limits)"""
    if isinstance(N, tuple):
        return m[0] <= N[0] and m[1] <= N[1]
    return sum(m) <= N

def trunc(f, N):
    """Drop every monomial of f that is not kept at order N (see keep)."""
    return P({m: co for m, co in f.dict().items() if keep(m, N)})

def to_P(ex):
    """Exact conversion SR -> P. chi^q becomes c^(18q); only integer powers of e0, Xi allowed."""
    ex = SR(ex)
    if ex.is_numeric():
        return P(QQ(ex))
    if ex.is_symbol():
        return SYM2P[ex]
    op, args = ex.operator(), ex.operands()
    if op in ADD:
        return sum(to_P(a) for a in args)
    if op in MUL:
        return prod(to_P(a) for a in args)
    if op is operator.pow:
        base, ee = args[0], QQ(args[1])
        if base == chi:
            k = 18 * ee
            assert k in ZZ, ex
            return P(c_r**ZZ(k))
        assert ee in ZZ and ee >= 0, ex
        return to_P(base)**ZZ(ee)
    raise ValueError("cannot convert %s" % ex)

def to_SR(f):
    """P -> SR, c -> chi^(1/18)."""
    out = 0
    for (i, j), co in f.dict().items():
        for (kc, ke, kx), q in co.dict().items():
            out += q * chi**(QQ(kc)/18) * eta**ke * x0**kx * e0**i * Xi**j
    return out

def unit_part(f):
    """Constant term of f; must be a pure power c^k. Returns k."""
    f0 = f.constant_coefficient()
    assert len(f0.monomials()) == 1 and f0.coefficients() == [1], f0
    kc, ke, kx = f0.exponents()[0]
    assert ke == 0 and kx == 0, f0
    return kc

def series_pow(f, b, N):
    """(f)^b truncated at order N (see keep), for f = c^k (1 + delta), delta = O(1)."""
    k = unit_part(f)
    kb = k * QQ(b)
    assert kb in ZZ, (k, b)
    delta = f * P(c_r**(-k)) - 1
    res, term = P(1), P(1)
    nmax = sum(N) if isinstance(N, tuple) else N   # delta^n has degree >= n
    for n in range(1, nmax + 1):
        term = trunc(term * delta, N) * (QQ(b) - n + 1) / n
        if term == 0:                               # everything further is truncated away
            break
        res += term
    return trunc(P(c_r**ZZ(kb)) * res, N)

def monomial_split(t):
    """t = coef * e0^a * e_t^b  ->  (coef, a, b) with a, b rational."""
    a = b = QQ(0)
    coef = SR(1)
    facs = t.operands() if t.operator() in MUL else [t]
    for fct in facs:
        if fct == e0:
            a += 1
        elif fct == e_t:
            b += 1
        elif fct.operator() is operator.pow and fct.operands()[0] == e0:
            a += QQ(fct.operands()[1])
        elif fct.operator() is operator.pow and fct.operands()[0] == e_t:
            b += QQ(fct.operands()[1])
        else:
            coef *= fct
    return coef, a, b

def xe_series(A, et_sub, taylor_order, N):
    """Series version of  A.taylor((e_t,0),(e0,0),taylor_order).subs(e_t = et_sub),
    truncated at order N (see keep).  et_sub is a P element divisible by e0."""
    u = P(et_sub // e0_s)                       # e_t = e0 * u
    assert u * e0_s == et_sub
    upow = {}
    out = P(0)
    terms = A.expand()
    terms = terms.operands() if terms.operator() in ADD else [terms]
    for t in terms:
        coef, a, b = monomial_split(t)
        if a + b > taylor_order:               # same cut as .taylor(..., taylor_order)
            continue
        assert (a + b) in ZZ and a + b >= 0, (a, b)
        if b not in upow:
            upow[b] = series_pow(u, b, N)
        out += trunc(to_P(coef) * e0_s**ZZ(a + b) * upow[b], N)
    return out

In [13]:
import time
pref_ps = P(c_r**-19) * e0_s            # e0 / chi^(19/18)

# ---- N:  et_e0_N = (xe_N^(19/12)).taylor(e0,0,5) * e0/chi^(19/18)
t_ps = time.time()
E0_ps  = to_P(et_e0_NLO_3PN.coefficient(Xi, 0))
xeN_ps = xe_series(xe_expr.coefficient(Xi,0).coefficient(x0), E0_ps, 4, 5)
etN_ps = trunc(series_pow(xeN_ps, 19/12, 5), 5) * pref_ps
print("N   series time:", time.time() - t_ps)
print("N   equal to et_e0_N:", to_P(et_e0_N.expand()) == etN_ps)
et_e0_N_ps = to_SR(etN_ps)

N   series time: 0.1159963607788086
N   equal to et_e0_N: True


In [14]:
# ---- 1PN: e_t -> et_e0_N + E2*Xi^2 inside the Xi^2 piece.
# Only Xi^0 and Xi^2 with e0 up to e0^5 are needed (et_e0_NNLO_1PN keeps the Xi^2 part),
# so truncate with separate limits (e0 <= 5, Xi <= 2) instead of total degree 7:
# no Xi^4, Xi^6 terms are ever generated.
O1 = (5, 2)
t_ps = time.time()
E2_ps   = to_P(et_e0_NLO_3PN.coefficient(Xi, 2))
xeN7_ps = xe_series(xe_expr.coefficient(Xi,0).coefficient(x0), E0_ps, 4, O1)
# this piece gets multiplied by Xi^2 below, so only its Xi^0 part can survive: order (5, 0)
xe1_ps  = xe_series(xe_expr.coefficient(Xi,2).coefficient(x0^2), etN_ps + E2_ps * Xi_s**2, 4, (5, 0))
xe_1PN_1_ps = trunc(xe1_ps * Xi_s**2 * P(x0_r), O1) + xeN7_ps
et1_ps = series_pow(xe_1PN_1_ps, 19/12, O1) * pref_ps      # * e0/chi^(19/18): e0 up to e0^6
print("1PN series time:", time.time() - t_ps)
print("Xi powers present:", sorted(set(m[1] for m in et1_ps.dict())))

# compare the Xi^2 part with the original (which also had Xi^4, Xi^6, ... from total degree 7)
xi2_ps = P({(i, 0): co for (i, j), co in et1_ps.dict().items() if j == 2})
print("1PN Xi^2 part equal to et_e0_1PN.coefficient(Xi,2):",
      to_P(et_e0_1PN.expand().coefficient(Xi, 2)) == xi2_ps)
et_e0_1PN_ps = to_SR(et1_ps)


1PN series time: 0.2673060894012451
1PN equal to et_e0_1PN: True


In [19]:
show(et_e0_1PN.expand())

4411380221/808455168*Xi^6*chi^(17/18)*e0*eta^3*x0^3 - 2851724129/269485056*Xi^6*chi^(5/18)*e0*eta^3*x0^3 - 63438782569/7545581568*Xi^6*chi^(17/18)*e0*eta^2*x0^3 + 1292068037/269485056*Xi^6*e0*eta^3*x0^3/chi^(7/18) + 41009819581/2515193856*Xi^6*chi^(5/18)*e0*eta^2*x0^3 + 912294776741/211276283904*Xi^6*chi^(17/18)*e0*eta*x0^3 + 267588055/808455168*Xi^6*e0*eta^3*x0^3/chi^(19/18) - 18580856593/2515193856*Xi^6*e0*eta^2*x0^3/chi^(7/18) - 2192359219/359313408*Xi^4*chi^(5/18)*e0^3*eta^2*x0^2 - 589750349609/70425427968*Xi^6*chi^(5/18)*e0*eta*x0^3 - 13119447220849/17747207847936*Xi^6*chi^(17/18)*e0*x0^3 - 549729485/1077940224*Xi^6*e0*eta^2*x0^3/chi^(19/18) + 1460196505/179656704*Xi^4*e0^3*eta^2*x0^2/chi^(7/18) + 267205922477/70425427968*Xi^6*e0*eta*x0^3/chi^(7/18) + 31527683591/5030387712*Xi^4*chi^(5/18)*e0^3*eta*x0^2 + 8481029139301/5915735949312*Xi^6*chi^(5/18)*e0*x0^3 + 3185814065/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(19/18) + 7905500665/30182326272*Xi^6*e0*eta*x0^3/chi^(19/18) - 327001429595/32697520128*Xi^4*e0^3*eta*x0^2/chi^(7/18) - 3842611057753/5915735949312*Xi^6*e0*x0^3/chi^(7/18) - 453390495499/281701711872*Xi^4*chi^(5/18)*e0^3*x0^2 - 659753/196992*Xi^4*chi^(5/18)*e0*eta^2*x0^2 - 3338788505/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(11/6) - 106691166739/9342148608*Xi^4*e0^3*eta*x0^2/chi^(19/18) - 113686717685/2535315406848*Xi^6*e0*x0^3/chi^(19/18) + 3150863507/479084544*Xi^2*e0^5*eta*x0/chi^(7/18) + 5479341746545/1831061127168*Xi^4*e0^3*x0^2/chi^(7/18) + 194045/98496*Xi^4*e0*eta^2*x0^2/chi^(7/18) + 3258933767/179656704*Xi^4*e0^3*eta^2*x0^2/chi^(5/2) + 9487717/2757888*Xi^4*chi^(5/18)*e0*eta*x0^2 + 170426008381/65395040256*Xi^4*e0^3*eta*x0^2/chi^(11/6) - 11147601665/479084544*Xi^2*e0^5*eta*x0/chi^(19/18) + 1845026319707/523160322048*Xi^4*e0^3*x0^2/chi^(19/18) + 271663/196992*Xi^4*e0*eta^2*x0^2/chi^(19/18) - 7092926885/359313408*Xi^4*e0^3*eta^2*x0^2/chi^(19/6) - 45311656423/13414367232*Xi^2*e0^5*x0/chi^(7/18) - 2790505/1378944*Xi^4*e0*eta*x0^2/chi^(7/18) - 252731074573/32697520128*Xi^4*e0^3*eta*x0^2/chi^(5/2) - 136440113/154441728*Xi^4*chi^(5/18)*e0*x0^2 + 4074514822747/3662122254336*Xi^4*e0^3*x0^2/chi^(11/6) + 3061519891285/174386774016*Xi^2*e0^5*x0/chi^(19/18) - 558101/393984*Xi^4*e0*eta*x0^2/chi^(19/18) + 102001329265/5030387712*Xi^4*e0^3*eta*x0^2/chi^(19/6) + 654631/131328*Xi^2*e0^3*eta*x0/chi^(7/18) + 40129445/77220864*Xi^4*e0*x0^2/chi^(7/18) + 575447233/79847424*Xi^2*e0^5*eta*x0/chi^(5/2) - 213231090871/261580161024*Xi^4*e0^3*x0^2/chi^(5/2) + 14528137028993/13144163549184*e0^7/chi^(19/18) - 1482433/131328*Xi^2*e0^3*eta*x0/chi^(19/18) + 8025889/22063104*Xi^4*e0*x0^2/chi^(19/18) + 3092267495/79847424*Xi^2*e0^5*eta*x0/chi^(19/6) - 1466851603085/281701711872*Xi^4*e0^3*x0^2/chi^(19/6) - 9414059/3677184*Xi^2*e0^3*x0/chi^(7/18) - 472055540597/29064462336*Xi^2*e0^5*x0/chi^(5/2) + 386822573/47803392*Xi^2*e0^3*x0/chi^(19/18) - 699589093187/29064462336*Xi^2*e0^5*x0/chi^(19/6) + 197/72*Xi^2*e0*eta*x0/chi^(7/18) + 173171/131328*Xi^2*e0^3*eta*x0/chi^(5/2) - 4108380839/479084544*Xi^2*e0^5*eta*x0/chi^(83/18) + 15994231/6653952*e0^5/chi^(19/18) - 197/72*Xi^2*e0*eta*x0/chi^(19/18) - 89629527689279/4381387849728*e0^7/chi^(19/6) + 654631/131328*Xi^2*e0^3*eta*x0/chi^(19/6) - 9901169371/479084544*Xi^2*e0^5*eta*x0/chi^(95/18) - 2833/2016*Xi^2*e0*x0/chi^(7/18) - 142057039/47803392*Xi^2*e0^3*x0/chi^(5/2) + 2706383369971/174386774016*Xi^2*e0^5*x0/chi^(83/18) + 2833/2016*Xi^2*e0*x0/chi^(19/18) - 9414059/3677184*Xi^2*e0^3*x0/chi^(19/6) + 142385851919/13414367232*Xi^2*e0^5*x0/chi^(95/18) + 3323/1824*e0^3/chi^(19/18) - 11042329/1108992*e0^5/chi^(19/6) + 140969652369941/4381387849728*e0^7/chi^(95/18) + e0/chi^(19/18) - 3323/1824*e0^3/chi^(19/6) + 50259743/6653952*e0^5/chi^(95/18) - 168548511070979/13144163549184*e0^7/chi^(133/18)

**1.5 PN**

I basically test if we can ignore substituting PN accurate e_t expressions into orders that are not going to contribute. And yes, you can ignore it here. expressions with _real do not ignore contributions from useless orders

In [15]:
xe_15_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_NLO_3PN.coefficient(Xi,3)*Xi^3})

In [16]:
xe_15_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:y})
#xe_15_1PN_real=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).subs({e_t:et_e0_N+et_e0_1PN*Xi^2})

In [17]:
xe_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
xe_15PN_2=xe_15_N+Xi^2*xe_15_1PN*x0+Xi^3*xe_15PN*x0^(3/2)
#xe_15PN_2_real=xe_15_N+Xi^2*xe_15_1PN_real*x0+Xi^3*xe_15PN*x0^(3/2)

et_e0_15PN=(((xe_15PN_2)^(19/12)).taylor((e0,0),(Xi,0),8))*(e0/chi^(19/18))
#et_e0_15PN_real=((xe_15PN_2_real)^(19/12)).taylor((e0,0),(Xi,0),6)*(e0/chi^(19/18))



In [ ]:
filename = "NNLO_e0/et_e0_NNLO_15PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_15PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_15PN =\n")
    f.write(str(et_e0_15PN.coefficient(Xi,3)*Xi^3))
    f.write("\n\n")

**2 PN**

In [ ]:
xe_2_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_NLO_3PN.coefficient(Xi,4)*Xi^4})
#To save computational time, newtonian order not important 

In [ ]:
xe_2_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2})
#xe_2_1PN_real=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).subs({e_t:et_e0_NLO_3PN.coefficient(Xi,0)+et_e0_NLO_3PN.coefficient(Xi,2)*Xi^2})

In [ ]:
xe_2_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e0:y})
#xe_2_15PN_real=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),0).subs({e0:et_e0_1PN})
#To save computational time, 1.5PN order not important 

In [ ]:
xe_2PN=xe_expr.coefficient(Xi,4).coefficient(x0^3).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
xe_2PN_1=xe_2PN*Xi^4*x0^2+xe_2_15PN*Xi^3*x0^(3/2)+xe_2_1PN*Xi^2*x0+xe_2_N
#xe_2PN_1_real=xe_2PN*Xi^4*x0^2+xe_2_15PN*Xi^3*x0^(3/2)+xe_2_1PN_real*Xi^2*x0+xe_2_N


et_e0_2PN=(((xe_2PN_1^(19/12))).taylor((e0,0),(Xi,0),9))*(e0/chi^(19/18))
#et_e0_2PN_real=(((xe_2PN_1_real^(19/12))).taylor((e0,0),(Xi,0),7))*(e0/chi^(19/18))

In [ ]:
filename = "NNLO_e0/et_e0_NNLO_2PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_2PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_2PN =\n")
    f.write(str(et_e0_2PN.coefficient(Xi,4)*Xi^4))
    f.write("\n\n")

In [ ]:
#show(et_e0_2PN.coefficient(Xi,4).expand().coefficient(e0,3)-et_e0_2PN_real.coefficient(Xi,4).expand().coefficient(e0,3))


**2.5 PN**

In [ ]:
xe_25_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_2PN.coefficient(Xi,4)*Xi^4+et_e0_NLO_3PN.coefficient(Xi,5)*Xi^5})

In [ ]:
update_progress("25_Pn_1")

In [ ]:
xe_25_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3})

In [ ]:
update_progress("25_Pn_2")

In [ ]:
xe_25_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_2PN.coefficient(Xi,4)*Xi^4})

In [ ]:
update_progress("25_Pn_3")

In [ ]:
xe_25_2PN=xe_expr.coefficient(Xi,4).coefficient(x0^3).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:y})
#xe_25_2PN_real=xe_expr.coefficient(Xi,4).coefficient(x0^3).taylor((e_t,0),(e0,0),0).subs({e_t:et_e0_1PN})

In [ ]:
update_progress("25_Pn_4")

In [ ]:
xe_25PN=xe_expr.coefficient(Xi,5).coefficient(x0^(7/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
update_progress("25_Pn_5")

In [ ]:
xe_25PN_1=(xe_25PN)*Xi^5*x0^(5/2)+xe_25_2PN*Xi^4*x0^2+xe_25_15PN*Xi^3*x0^(3/2)+xe_25_1PN*Xi^2*x0+xe_25_N
#xe_25PN_1_real=(xe_25PN)*Xi^5*x0^(5/2)+xe_25_2PN_real*Xi^4*x0^2+xe_25_15PN*Xi^3*x0^(3/2)+xe_25_1PN*Xi^2*x0+xe_25_N

et_e0_25PN=((((xe_25PN_1)^(19/12))).taylor((e0,0),(Xi,0),10))*(e0/chi^(19/18))
#et_e0_25PN_real=((((xe_25PN_1_real)^(19/12))).taylor((e0,0),(Xi,0),6))*(e0/chi^(19/18))

In [ ]:
filename = "NNLO_e0/et_e0_NNLO_25PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_25PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_25PN =\n")
    f.write(str(et_e0_25PN.coefficient(Xi,5)*Xi^5))
    f.write("\n\n")

**3 PN**

In [ ]:
xe_3_N=xe_expr.coefficient(Xi,0).coefficient(x0).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_2PN.coefficient(Xi,4)*Xi^4+et_e0_25PN.coefficient(Xi,5)*Xi^5+et_e0_NLO_3PN.coefficient(Xi,6)*Xi^6})

In [ ]:
update_progress("3_Pn_1")

In [ ]:
xe_3_1PN=xe_expr.coefficient(Xi,2).coefficient(x0^2).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_2PN.coefficient(Xi,4)*Xi^4})


In [ ]:
update_progress("3_Pn_2")

In [ ]:
xe_3_15PN=xe_expr.coefficient(Xi,3).coefficient(x0^(5/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3})

In [ ]:
update_progress("3_Pn_3")

In [ ]:
xe_3_2PN=xe_expr.coefficient(Xi,4).coefficient(x0^3).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2})


In [ ]:
update_progress("3_Pn_4")

In [ ]:
xe_3_25PN=xe_expr.coefficient(Xi,5).coefficient(x0^(7/2)).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e0:y})

In [ ]:
update_progress("3_Pn_5")

In [ ]:
xe_3PN=xe_expr.coefficient(Xi,6).coefficient(x0^4).taylor((e_t,0),(e0,0),4).collect(e_t).subs({e_t:et_e0_N})

In [ ]:
update_progress("3_Pn_6")

In [ ]:
xe_3PN_1=(xe_3PN)*Xi^6*x0^3+(xe_3_25PN)*Xi^5*x0^(5/2)+xe_3_2PN*Xi^4*x0^2+xe_3_15PN*Xi^3*x0^(3/2)+xe_3_1PN*Xi^2*x0+xe_3_N

et_e0_3PN=((((xe_3PN_1)^(19/12))).taylor((e0,0),(Xi,0),11))*(e0/chi^(19/18))


In [ ]:
update_progress("3_Pn_7")

In [ ]:
et_e0_NNLO_3PN=et_e0_N+et_e0_1PN.coefficient(Xi,2)*Xi^2+et_e0_15PN.coefficient(Xi,3)*Xi^3+et_e0_2PN.coefficient(Xi,4)*Xi^4+et_e0_25PN.coefficient(Xi,5)*Xi^5+et_e0_3PN.coefficient(Xi,6)*Xi^6

In [ ]:
filename = "NNLO_e0/et_e0_NNLO_3PN.txt"


with open(filename, "w") as f:
    f.write("# Expansion of et_e0_NNLO_3PN\n")
    f.write("# Plain ASCII output from Sage\n\n")
    f.write("et_e0_NNLO_3PN =\n")
    f.write(str(et_e0_NNLO_3PN))
    f.write("\n\n")